# Seminário 2: Geração e exploração de um banco de dados acerca dos docentes do ICMC-USP

## Autores

| Nome                                      | nUSP     |
| :---------------------------------------- | :------- |
| Lucas de Oliveira Ferreira                | 13695042 |
| Guilherme de Abreu Barreto                | 12543033 |
| Jhonathan Oliveira Alves                  | 11838116 |
| Lucas Pereira Franco de Almeida           | 12675020 |
| Miguel Prates Ferreira de Lima Cantanhede | 13672745 |


## Descrição

Neste notebook encontram-se todas as visualizações descritas em [nosso relatório](https://github.com/de-abreu/visualizacao_computacional/blob/main/seminario2/README.md), de tal forma que o leitor possa interatir com as mesmas. Para mais informações sobre as técnicas de visualização ou o contexto em que estas estão sendo empregadas, recomenda-se a leitura deste documento.

Para sua melhor visualização recomenda-se, após executar as células deste notebook, acessar as visualizações abrindo uma nova aba do navegador para as páginas web em que estas foram geradas. Os links para acessar estas encontram-se listadas abaixo:

- [Diagrama de arcos](http://127.0.0.1:8051/)
- [Gráfico de linhas](http://127.0.0.1:8052/)

Fizemos uso de três visualizações no total, descritas abaixo, cada qual para responder a dadas perguntas específicas:

### Diagrama de arcos

Quais projetos de pesquisa ou artigos científicos foram resultados da colaboração entre os pesquisadores do ICMC e,

- quais os professores mais colaborativos?
- com quem estes colaboram?
- quantas vezes estes já colaboraram?

### Gráfico de bolhas

Como grupos de pesquisa diferentes do ICMC se relacionam quando análisamos os títulos dos artigos publicados por seus membros docentes e,

- quais os grupos com mais e menos artigos?
- quais grupos possuem artigos que tratam de temas/áreas parecidas?


### Gráfico de linhas

Qual a produtividade dos docentes em função do tempo e,

- como esta se compara aos demais docentes deste mesmo instituto?
- quais são os artigos, por ano, de cada professor?


## Dependências

In [1]:
from sqlalchemy import create_engine
import pandas as pd


engine = create_engine("sqlite:///database/lattes.db") # Acesso ao banco de dados

## Diagrama de Arcos

In [2]:
from integrated_dashboard import create_integrated_dashboard

collaborations_query = """
WITH article_collaborations AS (
    SELECT
        r1.name AS researcher_1,
        r2.name AS researcher_2,
        a.title AS collaboration,
        'artigo' AS type,
        a.year AS start,
        a.year AS end
    FROM authorship au1
    JOIN authorship au2 ON au1.article_id = au2.article_id AND au1.author_id < au2.author_id
    JOIN researchers r1 ON au1.author_id = r1.lattes_id
    JOIN researchers r2 ON au2.author_id = r2.lattes_id
    JOIN articles a ON au1.article_id = a.id
),
project_collaborations AS (
    SELECT
        r1.name AS researcher_1,
        r2.name AS researcher_2,
        p.name AS collaboration,
        'projeto' AS type,
        p.start AS start,
        CAST(COALESCE(p.end, strftime('%Y', 'now')) AS INTEGER) AS end
    FROM participation p1
    JOIN participation p2 ON p1.project_id = p2.project_id AND p1.participant_id < p2.participant_id
    JOIN researchers r1 ON p1.participant_id = r1.lattes_id
    JOIN researchers r2 ON p2.participant_id = r2.lattes_id
    JOIN projects p ON p1.project_id = p.id
)
SELECT * FROM article_collaborations
UNION ALL
SELECT * FROM project_collaborations
"""

# Load dataframe with data extracted from the database query
try:
    collaborations: pd.DataFrame = pd.read_sql_query(collaborations_query, engine)
except Exception as e:
    print(f"✗ Erro ao carregar dados de colaboração do banco de dados: {e}")
    print(
        "  - Verifique se o banco de dados existe e contém as tabelas necessárias"
    )
    raise

print("✓ Dados de colaboração carregados com sucesso")
print(f"  - {len(collaborations)} registros carregados")

# Create and run the Dash app for the Arc Diagram visualization
app = create_integrated_dashboard(
    collab_df=collaborations,
    title="Colaborações entre Professores do ICMC, em artigos e projetos de pesquisa",
    legend_title="Colaborações",
)
app.run(host="127.0.0.1", port=8051, debug=True)

✓ Dados de colaboração carregados com sucesso
  - 807 registros carregados


## Gráfico de Bolhas

In [ ]:
from bubble_plot.bubble_plot import *

arts = load_articles(engine)
groups = load_groups()
macros_faltantes = load_macros_faltantes()

emb_articles, emb_groups, emb_macros = embeddings(arts, groups, macros_faltantes)

arts = get_groups(arts, emb_articles, groups, emb_groups)

XY = dim_reduction(emb_articles)
arts["x"], arts["y"] = XY[:, 0], XY[:, 1]

arts = map_macro(arts, emb_articles, macros_faltantes, emb_macros)

bubbles = create_bubble(arts)
plotting_bubbles(bubbles)

## Gráfico de Linhas

Para visualizar as duas colunas, utilizamos um código disponível no [stackoverflow](https://stackoverflow.com/questions/38783027/jupyter-notebook-display-two-pandas-tables-side-by-side)

In [ ]:
from IPython.display import display_html
from itertools import chain,cycle
def display_side_by_side(*args,titles=cycle([''])):
    html_str=''
    for df,title in zip(args, chain(titles,cycle(['</br>'])) ):
        html_str+='<th style="text-align:center"><td style="vertical-align:top">'
        html_str+=f'<h2 style="text-align: center;">{title}</h2>'
        html_str+=df.to_html().replace('table','table style="display:inline"')
        html_str+='</td></th>'
    display_html(html_str,raw=True)

In [ ]:
from line_graph.data import get_data
df,df_counts = get_data()


display_side_by_side(df.head(),df_counts.head(), titles=['df','df_counts']) #we left 3rd empty...

In [ ]:
from line_graph.dashboard import create_visualization

app = create_visualization(df,df_counts)

PORT = 8052
HOST = 'localhost'
print(f"{HOST}:{PORT}")
app.run(host=HOST, port=PORT)